## Transaction Level Metrics

In [1]:
import pandas as pd

from src.backend.input_handler import input_handling
from src.backend.market_data import get_current_prices

In [2]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

current_prices = get_current_prices(tickers)

[**********************88%*****************      ]  7 of 8 completed


In [3]:
transaction_data.merge(current_prices, how="left", on="Ticker").fillna(0.0)

,Portfolio,Transaction Date,Transaction Type,Ticker,Quantity,Transaction Price,Fee,Current Price
0,1,2023-03-10,BUY,VUAA.L,2,72.4600,0.00,148.360001
1,1,2023-04-11,BUY,VUAA.L,1,76.7000,0.00,148.360001
2,1,2023-05-10,BUY,VUAA.L,2,76.6800,0.00,148.360001
3,1,2023-06-09,BUY,VUAA.L,1,80.1800,0.00,148.360001
4,1,2023-07-14,BUY,VUAA.L,1,84.2600,0.00,148.360001
...,...,...,...,...,...,...,...,...
73,1,2026-05-12,SELL,XEON.DE,5,149.1054,2.10,150.184006
74,1,2026-06-11,SELL,XEON.DE,7,149.3430,2.14,150.184006
75,1,2026-07-10,SELL,XEON.DE,6,149.5995,2.12,150.184006
76,1,2024-08-06,BUY,SVXY,8,41.9000,2.15,62.880001


## Lot Calculation

In [16]:
import pandas as pd

from src.backend.input_handler import input_handling

In [22]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

In [23]:
# transaction_data = transaction_data.sort_values(by="Transaction Date")

transactions_dict = transaction_data.to_dict(orient='records')
transactions_dict[0:2]

[{'Portfolio': '1',
  'Transaction Date': Timestamp('2023-03-10 00:00:00'),
  'Transaction Type': 'BUY',
  'Ticker': 'VUAA.L',
  'Quantity': 2,
  'Transaction Price': 72.46,
  'Fee': 0.0},
 {'Portfolio': '1',
  'Transaction Date': Timestamp('2023-04-11 00:00:00'),
  'Transaction Type': 'BUY',
  'Ticker': 'VUAA.L',
  'Quantity': 1,
  'Transaction Price': 76.7,
  'Fee': 0.0}]

In [15]:
lots = []

for row in transactions_dict:
    if row.get("Transaction Type") == "BUY":
        lots.append({
            "Ticker": row.get("Ticker"),
            "Buy Date": row.get("Transaction Date"),
            "Initial Quantity": row.get("Quantity"),
            "Current Quantity": row.get("Quantity"),
            "Buy Price": row.get("Transaction Price"),
            "Buy Fee": row.get("Fee"),
            "Sell Fee": 0.0,
            "Realized PnL": 0.0,
            "Status": "OPEN",
        })
    elif row.get("Transaction Type") == "SELL":
        quantity_to_sell = row.get("Quantity")

        for lot in lots:
            if quantity_to_sell == 0:
                break

            if lot["Ticker"] == row.get("Ticker") and lot["Status"] == "OPEN":
                if lot["Current Quantity"] >= quantity_to_sell:
                    quantity_sold = quantity_to_sell
                else:
                    quantity_sold = lot["Current Quantity"]

                pro_rata_ratio = quantity_sold / row.get("Quantity")
                pro_rata_sell_fee = row.get("Fee") * pro_rata_ratio

                pro_rata_buy_fee = lot['Buy Fee'] * (quantity_sold / lot['Initial Quantity'])

                lot["Current Quantity"] -= quantity_sold
                lot["Sell Fee"] += pro_rata_sell_fee

                income = quantity_sold * row.get("Transaction Price")
                cost = quantity_sold * lot["Buy Price"]
                lot["Realized PnL"] += (income - cost - pro_rata_sell_fee - pro_rata_buy_fee)

                quantity_to_sell -= quantity_sold

                if lot["Current Quantity"] == 0:
                    lot["Status"] = "CLOSED"

pd.DataFrame(lots)

,Ticker,Buy Date,Initial Quantity,Current Quantity,Buy Price,Buy Fee,Sell Fee,Realized PnL,Status
0,VUSA.L,2022-08-26,1,1,66.480,0.00,0.0,0.0,OPEN
1,VUSA.L,2022-09-14,1,1,65.158,0.00,0.0,0.0,OPEN
2,VUSA.L,2022-10-18,1,1,62.430,0.00,0.0,0.0,OPEN
3,VUSA.L,2022-11-15,2,2,63.878,0.00,0.0,0.0,OPEN
4,VUSA.L,2022-12-16,2,2,60.163,0.00,0.0,0.0,OPEN
...,...,...,...,...,...,...,...,...,...
67,VUAA.L,2026-03-12,9,9,129.920,2.15,0.0,0.0,OPEN
68,VUAA.L,2026-04-20,4,4,136.520,2.07,0.0,0.0,OPEN
69,VUAA.L,2026-05-12,2,2,142.300,2.03,0.0,0.0,OPEN
70,VUAA.L,2026-06-11,4,4,141.400,2.07,0.0,0.0,OPEN


## Asset Level Calculation

In [43]:
import pandas as pd
import numpy as np
from pyxirr import xirr

from src.backend.input_handler import input_handling
from src.backend.market_data import get_current_prices
from src.backend.metrics_calculation import transaction_level_metrics, lot_level_metrics

In [2]:
filename = "real_transactions.csv"

transaction_data, tickers = input_handling(filename)

current_prices = get_current_prices(tickers)

transaction_data = transaction_level_metrics(transaction_data, current_prices)

lot_data = lot_level_metrics(transaction_data, current_prices)

[*********************100%***********************]  7 of 8 completed


In [21]:
lot_data_temp = lot_data.copy()

lot_data_temp['Cost Of Sold'] = (
    (lot_data_temp["Initial Quantity"] - lot_data_temp["Current Quantity"]) * lot_data_temp["Buy Price"]) + \
    (lot_data_temp["Buy Fee"] * ((lot_data_temp["Initial Quantity"] - lot_data_temp["Current Quantity"]) / lot_data_temp["Initial Quantity"])) + \
    lot_data_temp["Sell Fee"]

lot_data_temp['Cost Of Remaining'] = (
    lot_data_temp["Current Quantity"] * lot_data_temp["Buy Price"]) + \
    (lot_data_temp["Buy Fee"] * (lot_data_temp["Current Quantity"] / lot_data_temp["Initial Quantity"]))

lot_data_temp['Raw Remaining Cost'] = lot_data_temp["Current Quantity"] * lot_data_temp["Buy Price"]

aggs = {
    'Total Fees': ('Total Fees', 'sum'),
    'Total Cost': ('Total Cost', 'sum'),
    'Current Market Value': ('Current Market Value', 'sum'),
    'Realized PnL': ('Realized PnL', 'sum'),
    'Unrealized PnL': ('Unrealized PnL', 'sum'),
    'Current Quantity': ('Current Quantity', 'sum'),
    'Cost Of Remaining': ('Cost Of Remaining', 'sum'),
    'Cost Of Sold': ('Cost Of Sold', 'sum'),
    'Raw Remaining Cost': ('Raw Remaining Cost', 'sum')
}


asset_data = lot_data_temp.groupby("Ticker").agg(**aggs).reset_index(drop=False)

asset_data["Realized PnL %"] = ((asset_data['Realized PnL'] / asset_data['Cost Of Sold'].replace(0, np.nan)) * 100).replace(np.nan, 0)

asset_data["Unrealized PnL %"] = ((asset_data['Unrealized PnL'] / asset_data['Cost Of Remaining'].replace(0, np.nan)) * 100).replace(np.nan, 0)

asset_data['Total Return %'] = (((asset_data['Realized PnL'] + asset_data['Unrealized PnL']) / asset_data['Total Cost'].replace(0, np.nan)) * 100).replace(np.nan, 0)

asset_data['Average Entry Price'] = (asset_data['Raw Remaining Cost'] / asset_data['Current Quantity'].replace(0, np.nan)).replace(np.nan, 0)

asset_data['Percentage of Portfolio'] = ((asset_data['Current Market Value'] / asset_data['Current Market Value'].sum()) * 100 if asset_data['Current Market Value'].sum() != 0 else 0).replace(np.nan, 0)

asset_data

,Ticker,Total Fees,Total Cost,Current Market Value,Realized PnL,Unrealized PnL,Current Quantity,Cost Of Remaining,Cost Of Sold,Raw Remaining Cost,Realized PnL %,Unrealized PnL %,Total Return %,Average Entry Price,Percentage of Portfolio
0,AETF.AT,17.06,3257.050,4427.15004,0.000000,1170.10004,65,3257.050000,0.000000,3239.990,0.000000,35.925148,35.925148,49.846000,13.134961
1,DGRW.L,6.61,1028.230,1204.079956,0.000000,175.849956,24,1028.230000,0.000000,1021.620,0.000000,17.1022,17.1022,42.567500,3.572398
2,GRE.PA,0.00,40.560,88.724999,0.000000,48.164999,30,40.560000,0.000000,40.560,0.000000,118.749999,118.749999,1.352000,0.263239
3,SVXY,4.31,339.510,0.0,80.650000,0.0,0,0.000000,339.510000,0.000,23.754823,0,23.754823,0.000000,0.0
4,VHYD.L,27.77,5616.210,6296.940147,0.000000,680.730147,69,5616.210000,0.000000,5588.440,0.000000,12.12081,12.12081,80.991884,18.682462
5,VUAA.L,51.93,12864.950,16465.739594,0.000000,3600.789594,111,12864.950000,0.000000,12813.020,0.000000,27.989146,27.989146,115.432613,48.852387
6,VUSA.L,8.81,2166.481,2968.177574,0.000000,801.696574,27,2166.481000,0.000000,2157.671,0.000000,37.004551,37.004551,79.913741,8.806319
7,XEON.DE,15.58,6975.390,2254.275055,22.616611,27.464244,15,2226.810811,4748.579189,2225.700,0.476282,1.233344,0.717965,148.380000,6.688234


In [44]:
annualized_gain_pct = []

for row in asset_data.iterrows():
    row = row[1]
    asset_trx = transaction_data[transaction_data["Ticker"] == row["Ticker"]]
    dates = list(asset_trx["Transaction Date"])
    amounts = []

    for _, tx_row in asset_trx.iterrows():
        fee = tx_row.get('Fee', 0.0)
        if tx_row['Transaction Type'].upper() == 'BUY':
            amounts.append(-(tx_row['Quantity'] * tx_row['Transaction Price'] + fee))
        elif tx_row['Transaction Type'].upper() == 'SELL':
            amounts.append((tx_row['Quantity'] * tx_row['Transaction Price'] - fee))
            
    dates.append(pd.Timestamp.today())
    amounts.append(row["Current Market Value"])
    
    try:
        res = xirr(dates, amounts)
        annualized_gain_pct.append(res * 100 if res is not None else np.nan)
    except:
        annualized_gain_pct.append(np.nan)

asset_data['Annualized Gain %'] = annualized_gain_pct
asset_data

,Ticker,Total Fees,Total Cost,Current Market Value,Realized PnL,Unrealized PnL,Current Quantity,Cost Of Remaining,Cost Of Sold,Raw Remaining Cost,Realized PnL %,Unrealized PnL %,Total Return %,Average Entry Price,Percentage of Portfolio,Annualized Gain %
0,AETF.AT,17.06,3257.050,4427.15004,0.000000,1170.10004,65,3257.050000,0.000000,3239.990,0.000000,35.925148,35.925148,49.846000,13.134961,31.359633
1,DGRW.L,6.61,1028.230,1204.079956,0.000000,175.849956,24,1028.230000,0.000000,1021.620,0.000000,17.1022,17.1022,42.567500,3.572398,10.518440
2,GRE.PA,0.00,40.560,88.724999,0.000000,48.164999,30,40.560000,0.000000,40.560,0.000000,118.749999,118.749999,1.352000,0.263239,26.781926
3,SVXY,4.31,339.510,0.0,80.650000,0.0,0,0.000000,339.510000,0.000,23.754823,0,23.754823,0.000000,0.0,120.409917
4,VHYD.L,27.77,5616.210,6296.940147,0.000000,680.730147,69,5616.210000,0.000000,5588.440,0.000000,12.12081,12.12081,80.991884,18.682462,9.385636
5,VUAA.L,51.93,12864.950,16465.739594,0.000000,3600.789594,111,12864.950000,0.000000,12813.020,0.000000,27.989146,27.989146,115.432613,48.852387,19.834886
6,VUSA.L,8.81,2166.481,2968.177574,0.000000,801.696574,27,2166.481000,0.000000,2157.671,0.000000,37.004551,37.004551,79.913741,8.806319,14.542090
7,XEON.DE,15.58,6975.390,2254.275055,22.616611,27.464244,15,2226.810811,4748.579189,2225.700,0.476282,1.233344,0.717965,148.380000,6.688234,1.535082
